# 05 — Retrieval-Augmented Report Generation

**Phase 9, and the part of this project most likely to survive review.**

A decoder trained on 259 reports memorises phrasing, not medicine. Retrieval
sidesteps that: find the most visually similar historical studies, and build
the new report out of what radiologists actually wrote for those cases.

```
query frames ──► BiomedCLIP ──► embedding ──► vector index ──► top-K cases
                                                                    │
                                           ┌────────────────────────┤
                                           ▼                        ▼
                                   retrieved reports         query embedding
                                           └──────────┬─────────────┘
                                                      ▼
                                           section-aware fusion / LLM
                                                      ▼
                                              final clinical report
```

### Two rules that decide whether your numbers are real

1. **The index must never contain the query's own fold.** Retrieving from the
   full dataset means every test study finds itself, BLEU goes above 0.9, and
   the result is worthless. We rebuild the index per fold.
2. **Near-duplicates must be excluded too.** Notebook 02 found studies with
   near-identical reports (the same template for routine normals). If a
   duplicate of the query sits in the index, you are still copying the answer.
   We exclude by `dup_group`, not just by ID.

### Metrics

| Family | What it measures |
|---|---|
| Recall@K, MRR | does retrieval surface a clinically matching case? |
| BLEU / ROUGE-L / METEOR | surface overlap with the reference report |
| BERTScore | semantic similarity (optional, needs internet) |
| **Clinical efficacy P/R/F1** | does the generated text state the *right finding*? |

Clinical efficacy is the one that matters. A generated impression reading
"Normal USG findings." scores ~0.6 ROUGE-L against most references in this
dataset while being wrong for every abnormal case. Lead with clinical
efficacy in your results section and put BLEU in a supplementary table.

In [1]:
import os
import sys
import json
import re
import time
import subprocess
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

SEED = 42
np.random.seed(SEED)


def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


ART = Path("/kaggle/working/artifacts")
FIG = ART / "figures"
EMB = ART / "embeddings"
for d in (ART, FIG, EMB):
    d.mkdir(parents=True, exist_ok=True)


def find_file(fname):
    for base in [ART, EMB]:
        if (base / fname).exists():
            return base / fname
    for p in Path("/kaggle/input").rglob(fname):
        return p
    return None


master = find_file("master_split.csv")
assert master is not None, "Run notebooks 00-02 first."
df = pd.read_csv(master)
tiers = json.loads(find_file("label_tiers.json").read_text())
ontology = json.loads(find_file("ontology.json").read_text())
Ydf = pd.read_csv(find_file("labels_multihot.csv"))

TARGETS = [t for t in tiers["trainable"] + tiers["marginal"] if t in Ydf.columns]
Y = Ydf[TARGETS].values.astype(np.float32)
folds = df["fold"].values
study_ids = df["study_id"].values
dup_group = df["dup_group"].values if "dup_group" in df.columns else np.arange(len(df))
print(f"{len(df)} studies | {len(TARGETS)} targets | {len(set(dup_group))} duplicate groups")

259 studies | 13 targets | 259 duplicate groups


## 1. Load embeddings

Produced by notebook 04. If they are missing, a TF-IDF image-free fallback
lets you develop the rest of the pipeline, but do not report those numbers —
retrieval over report text is not image-to-report retrieval.

In [2]:
emb_path = find_file("study_image_embeddings.npy")
if emb_path is not None:
    Z = np.load(emb_path).astype(np.float32)
    RETRIEVAL_MODALITY = "BiomedCLIP image embedding"
    print("Loaded image embeddings:", Z.shape)
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    print("WARNING: no image embeddings found — using TF-IDF text fallback.\n"
          "         Development only. Run notebook 04 before reporting results.")
    V = TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True)
    X = V.fit_transform(df["findings_proc"].fillna(""))
    Z = TruncatedSVD(128, random_state=SEED).fit_transform(X).astype(np.float32)
    RETRIEVAL_MODALITY = "TF-IDF text fallback (NOT for publication)"

Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)

Loaded image embeddings: (259, 512)


## 2. Fold-aware, duplicate-aware retrieval

FAISS is used when available (and is what you should cite for scalability),
but at 259 studies an exact numpy inner-product search is identical in result
and simpler to audit. Both paths are exercised below so the code you publish
matches the code you ran.

In [3]:
try:
    import faiss
    HAS_FAISS = True
except ImportError:
    pipq("faiss-cpu")
    try:
        import faiss
        HAS_FAISS = True
    except ImportError:
        HAS_FAISS = False
print("FAISS:", HAS_FAISS)

K_MAX = 10


def build_index(vectors):
    if HAS_FAISS:
        idx = faiss.IndexFlatIP(vectors.shape[1])
        idx.add(np.ascontiguousarray(vectors))
        return idx
    return vectors


def search(index, queries, k):
    if HAS_FAISS:
        return index.search(np.ascontiguousarray(queries), k)
    sims = queries @ index.T
    order = np.argsort(-sims, axis=1)[:, :k]
    return np.take_along_axis(sims, order, 1), order


def retrieve_all(Z, k=K_MAX):
    """For every study, retrieve top-k from other folds, excluding its dup group."""
    neigh = np.full((len(Z), k), -1, dtype=int)
    sims = np.zeros((len(Z), k), dtype=np.float32)
    for f in sorted(set(folds)):
        te = np.where(folds == f)[0]
        pool = np.where(folds != f)[0]
        index = build_index(Z[pool])
        # over-fetch, then filter duplicates, then trim to k
        d, i = search(index, Z[te], min(len(pool), k + 25))
        for row, q in enumerate(te):
            keep = [(pool[j], s) for j, s in zip(i[row], d[row])
                    if j >= 0 and dup_group[pool[j]] != dup_group[q]]
            keep = keep[:k]
            for c, (j, s) in enumerate(keep):
                neigh[q, c] = j
                sims[q, c] = s
    return neigh, sims


t0 = time.time()
NEIGH, SIMS = retrieve_all(Z, K_MAX)
print(f"Retrieved top-{K_MAX} for {len(df)} studies in {time.time()-t0:.2f}s")

# Guard: nothing retrieved from the query's own fold or duplicate group.
assert all(folds[NEIGH[i, c]] != folds[i]
           for i in range(len(df)) for c in range(K_MAX) if NEIGH[i, c] >= 0), \
    "Fold leakage in retrieval."
assert all(dup_group[NEIGH[i, c]] != dup_group[i]
           for i in range(len(df)) for c in range(K_MAX) if NEIGH[i, c] >= 0), \
    "Duplicate leakage in retrieval."
print("Leakage guards passed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.8 MB/s eta 0:00:00
FAISS: True
Retrieved top-10 for 259 studies in 0.01s
Leakage guards passed.


## 3. Retrieval quality

A retrieval is **relevant** if the retrieved study shares at least one
disease label with the query. Reported as Recall@K (any relevant case in the
top K), Precision@K, and MRR.

The random baseline matters here: with "Normal study" at 32% prevalence,
random retrieval already gets Recall@5 near 0.8 for normal queries. Compare
against it or the number means nothing.

In [4]:
def relevant(i, j):
    return bool((Y[i] * Y[j]).sum() > 0)


def retrieval_metrics(neigh, ks=(1, 3, 5, 10)):
    out = {}
    valid = [i for i in range(len(df)) if Y[i].sum() > 0]
    for k in ks:
        rec, prec = [], []
        for i in valid:
            hits = [relevant(i, j) for j in neigh[i, :k] if j >= 0]
            rec.append(float(any(hits)))
            prec.append(np.mean(hits) if hits else 0.0)
        out[f"Recall@{k}"] = float(np.mean(rec))
        out[f"Precision@{k}"] = float(np.mean(prec))
    mrr = []
    for i in valid:
        r = next((c + 1 for c, j in enumerate(neigh[i]) if j >= 0 and relevant(i, j)), None)
        mrr.append(1.0 / r if r else 0.0)
    out["MRR"] = float(np.mean(mrr))
    return out


rng = np.random.default_rng(SEED)
rand_neigh = np.full_like(NEIGH, -1)
for f in sorted(set(folds)):
    te, pool = np.where(folds == f)[0], np.where(folds != f)[0]
    for i in te:
        cand = pool[dup_group[pool] != dup_group[i]]
        rand_neigh[i] = rng.choice(cand, K_MAX, replace=False)

ret_df = pd.DataFrame([
    {"method": RETRIEVAL_MODALITY, **retrieval_metrics(NEIGH)},
    {"method": "random baseline", **retrieval_metrics(rand_neigh)},
]).round(4)
ret_df.to_csv(ART / "retrieval_metrics.csv", index=False)
print(ret_df.to_string(index=False))

                    method  Recall@1  Precision@1  Recall@3  Precision@3  Recall@5  Precision@5  Recall@10  Precision@10    MRR
BiomedCLIP image embedding    0.3677       0.3677    0.6278       0.3438    0.7444       0.3256     0.8296        0.2960 0.5200
           random baseline    0.2018       0.2018    0.4664       0.2003    0.6143       0.1973     0.7848        0.1933 0.3699


## 4. Text metrics, implemented locally

BLEU, ROUGE-L and METEOR are implemented here rather than imported so the
notebook runs with internet off and so you can state the exact variant in
the paper (BLEU-4 with add-1 smoothing, corpus-level; ROUGE-L F-measure on
the longest common subsequence; METEOR unigram with a stem-match relaxation).
`nltk`/`rouge_score` are used instead when available, and the two are
cross-checked below.

In [5]:
def tokenize(s):
    return re.findall(r"[a-z0-9]+", str(s).lower())


def ngrams(toks, n):
    return Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))


def bleu(refs, hyps, max_n=4):
    """Corpus BLEU with add-1 smoothing on higher-order n-grams."""
    prec_num = [0] * max_n
    prec_den = [0] * max_n
    ref_len = hyp_len = 0
    for r, h in zip(refs, hyps):
        rt, ht = tokenize(r), tokenize(h)
        ref_len += len(rt)
        hyp_len += len(ht)
        for n in range(1, max_n + 1):
            rg, hg = ngrams(rt, n), ngrams(ht, n)
            prec_num[n - 1] += sum(min(c, rg[g]) for g, c in hg.items())
            prec_den[n - 1] += max(sum(hg.values()), 0)
    ps = []
    for n in range(max_n):
        num, den = prec_num[n], prec_den[n]
        if den == 0:
            ps.append(0.0)
        elif n == 0:
            ps.append(num / den)
        else:
            ps.append((num + 1) / (den + 1))          # smoothing
    if min(ps) <= 0:
        return {f"BLEU-{n+1}": 0.0 for n in range(max_n)}
    bp = 1.0 if hyp_len > ref_len else np.exp(1 - ref_len / max(hyp_len, 1))
    return {f"BLEU-{n+1}": float(bp * np.exp(np.mean(np.log(ps[:n + 1]))))
            for n in range(max_n)}


def lcs(a, b):
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        cur = [0] * (n + 1)
        for j in range(1, n + 1):
            cur[j] = prev[j - 1] + 1 if a[i - 1] == b[j - 1] else max(prev[j], cur[j - 1])
        prev = cur
    return prev[n]


def rouge_l(refs, hyps, beta=1.2):
    fs = []
    for r, h in zip(refs, hyps):
        rt, ht = tokenize(r), tokenize(h)
        l = lcs(rt, ht)
        if l == 0:
            fs.append(0.0)
            continue
        p, rc = l / len(ht), l / len(rt)
        fs.append(((1 + beta ** 2) * p * rc) / (rc + beta ** 2 * p))
    return float(np.mean(fs))


def _stem(w):
    for suf in ("ies", "es", "s", "ing", "ed"):
        if len(w) > 4 and w.endswith(suf):
            return w[: -len(suf)]
    return w


def meteor(refs, hyps, alpha=0.9, beta=3.0, gamma=0.5):
    scores = []
    for r, h in zip(refs, hyps):
        rt, ht = tokenize(r), tokenize(h)
        if not rt or not ht:
            scores.append(0.0)
            continue
        used, pairs = set(), []
        for i, w in enumerate(ht):                       # exact matches first
            for j, v in enumerate(rt):
                if j not in used and w == v:
                    used.add(j)
                    pairs.append((i, j))
                    break
        matched_h = {i for i, _ in pairs}
        for i, w in enumerate(ht):                       # then stem matches
            if i in matched_h:
                continue
            for j, v in enumerate(rt):
                if j not in used and _stem(w) == _stem(v):
                    used.add(j)
                    pairs.append((i, j))
                    break
        m = len(pairs)
        if m == 0:
            scores.append(0.0)
            continue
        p, rc = m / len(ht), m / len(rt)
        f = p * rc / (alpha * p + (1 - alpha) * rc)
        pairs.sort()
        chunks = 1 + sum(1 for k in range(1, len(pairs))
                         if pairs[k][1] != pairs[k - 1][1] + 1)
        scores.append(f * (1 - gamma * (chunks / m) ** beta))
    return float(np.mean(scores))


# Cross-check against reference implementations when they are installed.
try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    _r = [["the liver is normal in size".split()], ["gallstones are seen".split()]]
    _h = ["the liver is normal in size".split(), "the gallbladder shows stones".split()]
    ref_bleu = corpus_bleu(_r, _h, smoothing_function=SmoothingFunction().method1)
    mine = bleu(["the liver is normal in size", "gallstones are seen"],
                ["the liver is normal in size", "the gallbladder shows stones"])["BLEU-4"]
    print(f"BLEU-4 cross-check  nltk={ref_bleu:.4f}  local={mine:.4f}")
except ImportError:
    print("nltk not installed — using the local BLEU implementation "
          "(state 'add-1 smoothed corpus BLEU' in the paper).")

BLEU-4 cross-check  nltk=0.6580  local=0.6914


## 5. Clinical efficacy

Apply notebook 00's label rules to the **generated** text and compare the
extracted findings with those of the reference. This is the CheXpert-style
clinical-efficacy metric adapted to your ontology, and it is the metric a
clinical reviewer will care about.

In [6]:
def extract_labels(text):
    t = str(text).lower()
    out = set()
    for lab, spec in ontology.items():
        inc = spec.get("include", [])
        exc = spec.get("exclude", [])
        if any(re.search(p, t) for p in inc) and not any(re.search(p, t) for p in exc):
            out.add(lab)
    return out


ref_impressions = df["impression"].fillna("").values
ref_label_sets = [extract_labels(t) for t in ref_impressions]
print("Sanity: extractor assigns at least one label to "
      f"{np.mean([len(s) > 0 for s in ref_label_sets]):.0%} of reference impressions.")


def impression_of(text):
    """Score only the diagnostic clause, so long reports get no free label hits."""
    t = str(text)
    m = re.split(r"\b(?:IMPRESSION|COMMENTS?)\s*[:\-]", t, flags=re.I)
    return m[-1] if len(m) > 1 else t


def clinical_efficacy(hyps, ref_sets):
    tp = fp = fn = 0
    per_label = defaultdict(lambda: [0, 0, 0])
    exact = 0
    for h, rs in zip(hyps, ref_sets):
        hs = extract_labels(impression_of(h))
        exact += int(hs == rs)
        for l in hs & rs:
            tp += 1
            per_label[l][0] += 1
        for l in hs - rs:
            fp += 1
            per_label[l][1] += 1
        for l in rs - hs:
            fn += 1
            per_label[l][2] += 1
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {
        "CE_precision": p,
        "CE_recall": r,
        "CE_F1": 2 * p * r / (p + r) if p + r else 0.0,
        "CE_exact_set": exact / len(hyps),
    }, per_label

Sanity: extractor assigns at least one label to 98% of reference impressions.


## 6. Generation strategies

### 6a. Retrieve-and-copy (K=1)
Copy the nearest neighbour's impression verbatim. Deceptively strong, and the
baseline every retrieval paper must report.

### 6b. Consensus fusion over top-K
Extract the finding set from each of the K retrieved impressions, keep
findings that appear in at least `VOTE_FRAC` of them (similarity-weighted),
then realise them as a report using each finding's most common phrasing in
the training folds. Abstains to "Normal study" only when the retrieved
consensus is genuinely normal.

### 6c. Section-aware fusion
Same voting, but the *findings* paragraph is assembled organ-by-organ from
the retrieved reports rather than copied from one. This is the
"section-aware report encoding" contribution made concrete.

In [7]:
K_USE = 5
VOTE_FRAC = 0.4

impressions = df["impression"].fillna("").values
findings_full = df["findings"].fillna("").values


def weighted_vote(i, k=K_USE, frac=VOTE_FRAC):
    """Similarity-weighted vote over the labels of the top-k retrieved cases."""
    idx = [j for j in NEIGH[i, :k] if j >= 0]
    if not idx:
        return set(), []
    w = np.clip(SIMS[i, :len(idx)], 1e-6, None)
    w = w / w.sum()
    score = defaultdict(float)
    for wt, j in zip(w, idx):
        for l in ref_label_sets[j]:
            score[l] += wt
    return {l for l, s in score.items() if s >= frac}, idx


# Canonical phrasing for each label = its most frequent reference impression.
def build_phrasebook(train_idx):
    book = {}
    for lab in ontology:
        cands = [impressions[j] for j in train_idx if lab in ref_label_sets[j]
                 and len(ref_label_sets[j]) == 1]
        if cands:
            book[lab] = Counter(c.strip() for c in cands).most_common(1)[0][0]
    return book


gen_copy, gen_fuse, gen_section = [], [], []
for i in range(len(df)):
    train_idx = np.where(folds != folds[i])[0]
    book = build_phrasebook(train_idx)

    # 6a — copy top-1
    top = NEIGH[i, 0]
    gen_copy.append(impressions[top] if top >= 0 else "Normal USG findings.")

    # 6b — consensus fusion
    labs, idx = weighted_vote(i)
    if not labs:
        gen_fuse.append(book.get("Normal study", "Normal USG findings."))
    else:
        parts = [book.get(l) for l in sorted(labs) if book.get(l)]
        parts = parts or [impressions[idx[0]]]
        if len(parts) == 1:
            gen_fuse.append(parts[0])
        else:
            gen_fuse.append("Suggestive of " + "; ".join(
                re.sub(r"^(suggestive of|features of)\s+", "", p.strip().rstrip("."),
                       flags=re.I) for p in parts) + ".")

    # 6c — section-aware: organ lines voted across retrieved findings sections
    organ_lines = defaultdict(Counter)
    for j in idx:
        for line in re.split(r"(?<=[.])\s+", findings_full[j]):
            m = re.match(r"\s*([A-Z][A-Za-z .\-/]{2,28}?)\s*[:\-]\s*(.+)", line)
            if m:
                organ_lines[m.group(1).strip().title()][m.group(2).strip()] += 1
    body = []
    for organ, ctr in organ_lines.items():
        if sum(ctr.values()) >= max(2, int(0.4 * max(len(idx), 1))):
            body.append(f"{organ}: {ctr.most_common(1)[0][0]}")
    gen_section.append(
        (" ".join(body[:12]) + " IMPRESSION: " + gen_fuse[i]).strip())

print("Example generations")
for i in [0, 1, 4]:
    print(f"\n--- {study_ids[i]}   (labels: {sorted(ref_label_sets[i])})")
    print(f"  REF    : {impressions[i][:130]}")
    print(f"  copy@1 : {gen_copy[i][:130]}")
    print(f"  fusion : {gen_fuse[i][:130]}")

Example generations

--- R1   (labels: ['Inhomogeneous uterus'])
  REF    : Suggestive of Inhomogeneous uterus.
  copy@1 : Normal study at ultrasound.
  fusion : Normal USG findings.

--- R2   (labels: ['Ovarian cyst'])
  REF    : Right adnexal chocolate cyst.
  copy@1 : 1. Cholelithiasis. 2. Significant PVR.
  fusion : Moderate PVR.

--- R5   (labels: ['Ovarian cyst'])
  REF    : Suggestive of Enlarged right ovary with endometrioma.
  copy@1 : Suggestive of 1. Cholelithiasis, 2. Hepatomegaly.
  fusion : Normal USG findings.


### 6d. LLM rewriting (optional)

Feeds the retrieved reports to a small instruction-tuned model as context.
Off by default: on Kaggle it needs internet plus a GPU, and on 259 studies it
rarely beats consensus fusion on clinical efficacy while being far harder to
reproduce. Turn it on for one ablation row, not for the headline result.

In [8]:
RUN_LLM = False
LLM_NAME = "google/flan-t5-base"
gen_llm = None

if RUN_LLM:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        tok = AutoTokenizer.from_pretrained(LLM_NAME)
        lm = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        lm = lm.to(dev).eval()
        gen_llm = []
        for i in range(len(df)):
            idx = [j for j in NEIGH[i, :K_USE] if j >= 0]
            ctx = "\n".join(f"Case {c+1}: {impressions[j]}" for c, j in enumerate(idx))
            prompt = ("You are a radiologist. Below are impressions from the most "
                      "visually similar previous ultrasound studies.\n" + ctx +
                      "\nWrite the single most likely impression for the new study, "
                      "in the same style, in one or two sentences.")
            ids = tok(prompt, return_tensors="pt", truncation=True,
                      max_length=512).to(dev)
            with torch.no_grad():
                out = lm.generate(**ids, max_new_tokens=60, num_beams=4)
            gen_llm.append(tok.decode(out[0], skip_special_tokens=True))
            if i % 50 == 0:
                print(f"  {i}/{len(df)}", end="\r")
        print("\nLLM rewriting done.")
    except Exception as e:
        print("LLM rewriting skipped:", type(e).__name__, e)
        gen_llm = None

## 7. Evaluation

Includes two degenerate baselines. If "always say normal" beats your system
on BLEU — and it often will — that is the finding that justifies leading with
clinical efficacy instead.

In [9]:
CANDIDATES = {
    "constant_normal": ["Normal USG findings."] * len(df),
    "random_train_report": [impressions[j] for j in rand_neigh[:, 0]],
    "retrieve_copy@1": gen_copy,
    "consensus_fusion@K": gen_fuse,
}
mt_gen = find_file("generated_multitask.csv")
if mt_gen is not None:
    g = pd.read_csv(mt_gen).set_index("study_id")["generated"].to_dict()
    CANDIDATES["multitask_decoder"] = [str(g.get(s, "")) for s in study_ids]
if gen_llm is not None:
    CANDIDATES["rag_llm_rewrite"] = gen_llm

rows, per_label_store = [], {}
for name, hyps in CANDIDATES.items():
    m = bleu(ref_impressions, hyps)
    m["ROUGE-L"] = rouge_l(ref_impressions, hyps)
    m["METEOR"] = meteor(ref_impressions, hyps)
    ce, per_lab = clinical_efficacy(hyps, ref_label_sets)
    m.update(ce)
    m["mean_len"] = float(np.mean([len(tokenize(h)) for h in hyps]))
    m["distinct"] = len(set(hyps)) / len(hyps)
    rows.append({"method": name, **m})
    per_label_store[name] = per_lab

gen_df = pd.DataFrame(rows).round(4)
cols = ["method", "CE_F1", "CE_precision", "CE_recall", "CE_exact_set",
        "BLEU-1", "BLEU-4", "ROUGE-L", "METEOR", "distinct", "mean_len"]
gen_df = gen_df[[c for c in cols if c in gen_df.columns]]
gen_df = gen_df.sort_values("CE_F1", ascending=False)
gen_df.to_csv(ART / "generation_metrics.csv", index=False)
print(gen_df.to_string(index=False))

pd.DataFrame({"study_id": study_ids, "reference": ref_impressions,
              **{k: v for k, v in CANDIDATES.items()}}).to_csv(
    ART / "generated_reports.csv", index=False)

             method  CE_F1  CE_precision  CE_recall  CE_exact_set  BLEU-1  BLEU-4  ROUGE-L  METEOR  distinct  mean_len
 consensus_fusion@K 0.4108        0.4373     0.3873        0.3745  0.1611  0.1002   0.2282  0.1890    0.0849    4.5985
    constant_normal 0.3171        0.3514     0.2889        0.3243  0.0000  0.0000   0.1590  0.1184    0.0039    3.0000
    retrieve_copy@1 0.2813        0.2741     0.2889        0.2510  0.2792  0.1937   0.2340  0.2309    0.3784    7.5405
  multitask_decoder 0.2192        0.1584     0.3556        0.0849  0.0943  0.0375   0.1400  0.2241    0.9614   32.9691
random_train_report 0.1531        0.1508     0.1556        0.1313  0.1166  0.0456   0.1192  0.0952    0.4517    7.7104


### Full-report track

The section-aware method writes a whole report, so it must not be scored
against a one-line impression — a 70-word report is guaranteed to lose on
BLEU to a 5-word one regardless of quality. Compare full reports with full
reports.

In [11]:
ref_full = [f"{f.strip()} IMPRESSION: {i.strip()}"
            for f, i in zip(findings_full, impressions)]
FULL_CANDIDATES = {
    "copy_full_report@1": [
        f"{findings_full[j].strip()} IMPRESSION: {impressions[j].strip()}"
        if j >= 0 else "Normal USG findings." for j in NEIGH[:, 0]],
    "section_aware_fusion": gen_section,
}
frows = []
for name, hyps in FULL_CANDIDATES.items():
    m = bleu(ref_full, hyps)
    m["ROUGE-L"] = rouge_l(ref_full, hyps)
    m["METEOR"] = meteor(ref_full, hyps)
    ce, _ = clinical_efficacy(hyps, ref_label_sets)
    m.update(ce)
    m["mean_len"] = float(np.mean([len(tokenize(h)) for h in hyps]))
    frows.append({"method": name, **m})
full_df = pd.DataFrame(frows).round(4)
full_df = full_df[[c for c in cols if c in full_df.columns]]
full_df.to_csv(ART / "generation_metrics_fullreport.csv", index=False)
print("Full-report track (reference = findings + impression):")
print(full_df.to_string(index=False))

Full-report track (reference = findings + impression):
              method  CE_F1  CE_precision  CE_recall  CE_exact_set  BLEU-1  BLEU-4  ROUGE-L  METEOR  mean_len
  copy_full_report@1 0.2813        0.2741     0.2889        0.2510  0.6116  0.4585   0.5357  0.5581  156.1853
section_aware_fusion 0.4108        0.4373     0.3873        0.3745  0.2702  0.1650   0.3219  0.2953   81.2239


### BERTScore (optional)

Needs internet. Use `microsoft/deberta-xlarge-mnli` for the published number
if you can afford it; `distilbert-base-uncased` is a reasonable fallback.

In [12]:
RUN_BERTSCORE = False
if RUN_BERTSCORE:
    try:
        from bert_score import score as bscore
    except ImportError:
        pipq("bert-score")
        try:
            from bert_score import score as bscore
        except ImportError:
            bscore = None
    if bscore is not None:
        bs_rows = []
        for name, hyps in CANDIDATES.items():
            P, R, F = bscore(list(hyps), list(ref_impressions), lang="en",
                             model_type="distilbert-base-uncased", verbose=False)
            bs_rows.append({"method": name, "BERTScore_F1": float(F.mean())})
        bs = pd.DataFrame(bs_rows).round(4)
        bs.to_csv(ART / "bertscore.csv", index=False)
        print(bs.to_string(index=False))

## 8. Ablation: how many neighbours?

The K sweep is a required ablation for a retrieval paper, and it usually has
a clear story: small K copies noise, large K regresses to "everything is
normal" because normals dominate the pool.

In [13]:
sweep = []
for k in [1, 2, 3, 5, 7, 10]:
    hyps = []
    for i in range(len(df)):
        train_idx = np.where(folds != folds[i])[0]
        book = build_phrasebook(train_idx)
        labs, idx = weighted_vote(i, k=k)
        if not labs:
            hyps.append(book.get("Normal study", "Normal USG findings."))
        else:
            parts = [book.get(l) for l in sorted(labs) if book.get(l)] or \
                    [impressions[idx[0]]]
            hyps.append(parts[0] if len(parts) == 1 else "Suggestive of " + "; ".join(
                re.sub(r"^(suggestive of|features of)\s+", "", p.strip().rstrip("."),
                       flags=re.I) for p in parts) + ".")
    ce, _ = clinical_efficacy(hyps, ref_label_sets)
    r = retrieval_metrics(NEIGH, ks=(k,))
    sweep.append({"K": k, **ce, "ROUGE-L": rouge_l(ref_impressions, hyps),
                  "Recall@K": r[f"Recall@{k}"], "Precision@K": r[f"Precision@{k}"]})
sw = pd.DataFrame(sweep).round(4)
sw.to_csv(ART / "rag_k_sweep.csv", index=False)
print(sw.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(sw["K"], sw["CE_F1"], "o-", label="Clinical efficacy F1")
ax[0].plot(sw["K"], sw["CE_precision"], "s--", label="CE precision")
ax[0].plot(sw["K"], sw["CE_recall"], "^--", label="CE recall")
ax[0].set_xlabel("K retrieved cases")
ax[0].set_ylabel("score")
ax[0].set_title("Effect of K on clinical efficacy")
ax[0].legend(fontsize=8)
ax[0].grid(alpha=0.3)

g = gen_df.set_index("method")
g[["CE_F1", "ROUGE-L"]].plot.barh(ax=ax[1], width=0.75)
ax[1].set_title("Generation methods: clinical vs surface metrics")
ax[1].invert_yaxis()
ax[1].grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / "rag_results.png")
plt.show()

 K  CE_precision  CE_recall  CE_F1  CE_exact_set  ROUGE-L  Recall@K  Precision@K
 1        0.2964     0.2889 0.2926        0.2587   0.1877    0.3677       0.3677
 2        0.2676     0.4095 0.3237        0.1467   0.2240    0.5291       0.3498
 3        0.4176     0.3460 0.3785        0.3629   0.2000    0.6278       0.3438
 5        0.4373     0.3873 0.4108        0.3745   0.2282    0.7444       0.3256
 7        0.4203     0.3683 0.3926        0.3514   0.2045    0.7982       0.3120
10        0.4259     0.3556 0.3875        0.3784   0.1972    0.8296       0.2960


## 9. Qualitative case studies

Pick two successes and two failures for the paper. Reviewers of medical
report-generation work read these before they read your tables, and an
honest failure case buys more credibility than four pages of metrics.

In [14]:
best = gen_df.iloc[0]["method"]
hyps = CANDIDATES[best]
ok, bad = [], []
for i in range(len(df)):
    hs, rs = extract_labels(impression_of(hyps[i])), ref_label_sets[i]
    (ok if hs == rs and rs else bad).append(i)
print(f"Best method: {best} | exact-set correct on {len(ok)}/{len(df)}")

cases = []
for i in (ok[:2] + bad[:2]):
    cases.append({
        "study_id": study_ids[i],
        "outcome": "correct" if i in ok else "incorrect",
        "reference": impressions[i],
        "generated": hyps[i],
        "ref_labels": "|".join(sorted(ref_label_sets[i])),
        "gen_labels": "|".join(sorted(extract_labels(impression_of(hyps[i])))),
        "top_neighbours": "|".join(str(study_ids[j]) for j in NEIGH[i, :3] if j >= 0),
        "top_sims": "|".join(f"{s:.3f}" for s in SIMS[i, :3]),
    })
cases_df = pd.DataFrame(cases)
cases_df.to_csv(ART / "rag_case_studies.csv", index=False)
for c in cases:
    print(f"\n[{c['outcome']}] {c['study_id']}  (neighbours: {c['top_neighbours']})")
    print(f"  REF: {c['reference'][:140]}")
    print(f"  GEN: {impression_of(c['generated'])[:140]}")

Best method: consensus_fusion@K | exact-set correct on 97/259

[correct] R4  (neighbours: R208|R218|R146)
  REF: Suggestive of early pregnancy failure (Missed abortion).
  GEN: Suggestive of early pregnancy failure(Missed abortion).

[correct] R6  (neighbours: R177|R69|R57)
  REF: Normal USG findings.
  GEN: Normal USG findings.

[incorrect] R1  (neighbours: R7|R126|R64)
  REF: Suggestive of Inhomogeneous uterus.
  GEN: Normal USG findings.

[incorrect] R2  (neighbours: R114|R105|R35)
  REF: Right adnexal chocolate cyst.
  GEN: Moderate PVR.


In [15]:
summary = {
    "retrieval_modality": RETRIEVAL_MODALITY,
    "faiss": HAS_FAISS,
    "K_used": K_USE,
    "vote_fraction": VOTE_FRAC,
    "best_method": best,
    "best_CE_F1": float(gen_df.iloc[0]["CE_F1"]),
    "recall@5": float(ret_df.iloc[0]["Recall@5"]),
    "MRR": float(ret_df.iloc[0]["MRR"]),
}
(ART / "rag_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "retrieval_modality": "BiomedCLIP image embedding",
  "faiss": true,
  "K_used": 5,
  "vote_fraction": 0.4,
  "best_method": "consensus_fusion@K",
  "best_CE_F1": 0.4108,
  "recall@5": 0.7444,
  "MRR": 0.52
}


## What to carry into the paper

- **Table 3** = `generation_metrics.csv`, with `constant_normal` kept in.
- **Table 4** = `retrieval_metrics.csv` with the random baseline.
- **Figure** = the K sweep. Explain the turning point mechanistically.
- **Case studies** = `rag_case_studies.csv`, two good and two bad.

The claim you can defend: *retrieval-augmented fusion improves clinical
efficacy over both a fine-tuned decoder and nearest-neighbour copying, on a
small single-centre ultrasound corpus.* Do not claim clinical readiness —
with 259 studies from one machine and one reporting style, external validity
is the obvious limitation, and stating it first disarms the reviewer who was
going to raise it.

Next: **06 — explainability, ablations and statistical validation.**